In [1]:
!git clone https://github.com/MR-just01/Llama3.2-Reasoning

%cd Llama3.2-Reasoning

!find . -maxdepth 3 -type f | head -50

Cloning into 'Llama3.2-Reasoning'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 134 (delta 73), reused 42 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 22.01 MiB | 14.60 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/kaggle/working/Llama3.2-Reasoning
./.gitignore
./data/raw/ReadME.md
./data/splits/ReadME.md
./data/processed/reasoning_dataset (1).csv
./data/processed/aqua_rat_standardized.csv
./data/processed/validation_results.csv
./data/processed/arc_challenge_standardized.csv
./data/processed/startegyQA_standardized.csv
./data/processed/gsm8k_standardized.csv
./LICENSE
./requirements.txt
./.git/packed-refs
./.git/index
./.git/HEAD
./.git/description
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/post-update.sample
./.git/hooks/pre-receive.sample
./.git/hooks/applypatch-msg.sample
./.git/hooks/pre-rebase.sa

In [2]:
import pandas as pd

validation_df = pd.read_csv(
    "./data/processed/validation_results.csv"
)

print("Rows:", len(validation_df))
print("Columns:")
print(validation_df.columns.tolist())

display(validation_df.head())

Rows: 3000
Columns:
['index', 'instruction', 'input', 'expected_answer', 'model_response', 'generation_time', 'dataset', 'task_type']


,index,instruction,input,expected_answer,model_response,generation_time,dataset,task_type
0,3222,Solve the following math reasoning problem ste...,"Before he lost one, Policeman O'Brien had 5 mo...",34,Reasoning:\nTwice as many hats as fire chief S...,8.662204,gsm8k,math_reasoning
1,3160,Solve the following multiple-choice math reaso...,Find the area of trapezium whose parallel side...,228 cm2,Reasoning:\nArea of trapezium = 1/2 * (sum of ...,6.628668,AQUA-RAT,math_reasoning
2,27604,Solve the following multiple-choice math reaso...,How many positive three-digit integers are div...,43,Reasoning:\nThe number must be divisible by 21...,8.098332,AQUA-RAT,math_reasoning
3,11816,Solve the following math reasoning problem ste...,Ten more than twice the number of birds on the...,20,Reasoning:\nLet x be the number of birds on th...,6.906947,gsm8k,math_reasoning
4,25341,Solve the following math reasoning problem ste...,Tod drives his family car 55 miles to the nort...,6,Reasoning:\nFirst find the total distance Tod ...,5.802851,gsm8k,math_reasoning


In [15]:
judge_df = df.copy()

print("Original rows:", len(df))
print("Judge rows:", len(judge_df))

Original rows: 3000
Judge rows: 3000


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("No GPU detected.")

CUDA available: True
GPU: Tesla T4
GPU count: 2
GPU memory: 14.56 GB


In [3]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 52.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 35.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.7 MB/s eta 0:00:00


In [5]:
!pip install -q groq

In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret(
    "GROQ_API_KEY"
)

print("API key loaded:", bool(GROQ_API_KEY))

API key loaded: True


In [13]:
# from groq import Groq

# client = Groq(
#     api_key=GROQ_API_KEY
# )

# print("Groq client initialized.")

Groq client initialized.


In [8]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "What is 15 × 3? Return only the number."
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

45


In [10]:
import transformers
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

Transformers: 5.14.1
Accelerate: 1.14.0
bitsandbytes: 0.50.0


In [5]:
from huggingface_hub import login

login()

In [11]:
print(validation_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            3000 non-null   int64  
 1   instruction      3000 non-null   object 
 2   input            3000 non-null   object 
 3   expected_answer  2998 non-null   object 
 4   model_response   3000 non-null   object 
 5   generation_time  3000 non-null   float64
 6   dataset          3000 non-null   object 
 7   task_type        3000 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 187.6+ KB
None


In [23]:
import os

print("GROQ_API_KEY exists:",
      "GROQ_API_KEY" in os.environ)

print("Possible Groq variables:")

for key in os.environ:
    if "GROQ" in key.upper():
        print(key)

GROQ_API_KEY exists: False
Possible Groq variables:


In [22]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

print("✅ Groq API key loaded successfully")
print("Key length:", len(GROQ_API_KEY))

✅ Groq API key loaded successfully
Key length: 56


In [31]:
# ============================================================
# FINAL LLAMA EVALUATION PIPELINE
# CELL 1 — SETUP + FIXED STRATIFIED 300-ROW SAMPLE
# ============================================================

import os
import json
import time
import math
import random
import numpy as np
import pandas as pd
from groq import Groq


# ============================================================
# CONFIG
# ============================================================

GROQ_MODEL = "openai/gpt-oss-120b"

SAMPLE_SIZE = 300
CALIBRATION_SIZE = 10

RANDOM_STATE = 42

CHECKPOINT_PATH = (
    "/kaggle/working/llama_groq_final_300_checkpoint.csv"
)

SAMPLE_PATH = (
    "/kaggle/working/llama_groq_final_300_sample.csv"
)

CALIBRATION_PATH = (
    "/kaggle/working/llama_groq_calibration_10.csv"
)

# High reasoning effort for calibration.
CALIBRATION_REASONING_EFFORT = "high"

# We will decide the full-evaluation effort after
# manually auditing the calibration results.
FULL_REASONING_EFFORT = "medium"

MAX_RETRIES = 3
INITIAL_RETRY_WAIT = 5

MAX_COMPLETION_TOKENS = 1200

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# ============================================================
# GROQ CLIENT
# ============================================================

# GROQ_API_KEY was loaded previously from Kaggle Secrets.

client = Groq(
    api_key=GROQ_API_KEY
)

print("✅ Groq client initialized")
print("Judge model:", GROQ_MODEL)


# ============================================================
# PREPARE EVALUATION DATA
# ============================================================

evaluation_df = validation_df[
    validation_df["expected_answer"].notna()
    &
    validation_df["input"].notna()
    &
    validation_df["model_response"].notna()
].copy()

evaluation_df["index"] = (
    evaluation_df["index"].astype(str)
)

print()
print("=" * 80)
print("ELIGIBLE EVALUATION ROWS:", len(evaluation_df))
print("=" * 80)


# ============================================================
# BASIC SAFETY CHECK
# ============================================================

if len(evaluation_df) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(evaluation_df)} eligible rows are available, "
        f"but SAMPLE_SIZE={SAMPLE_SIZE}."
    )


# ============================================================
# CREATE / LOAD FIXED STRATIFIED SAMPLE
# ============================================================

if os.path.exists(SAMPLE_PATH):

    # --------------------------------------------------------
    # Load previously created fixed sample
    # --------------------------------------------------------

    sample_df = pd.read_csv(SAMPLE_PATH)

    sample_df["index"] = (
        sample_df["index"].astype(str)
    )

    print("\nExisting fixed sample loaded.")
    print("Sample rows:", len(sample_df))


else:

    # --------------------------------------------------------
    # Determine stratification variable
    # --------------------------------------------------------

    if "task_type" in evaluation_df.columns:

        evaluation_df["_stratum"] = (
            evaluation_df["task_type"]
            .fillna("UNKNOWN")
            .astype(str)
        )

        stratification_used = "task_type"

    elif "dataset" in evaluation_df.columns:

        evaluation_df["_stratum"] = (
            evaluation_df["dataset"]
            .fillna("UNKNOWN")
            .astype(str)
        )

        stratification_used = "dataset"

    else:

        evaluation_df["_stratum"] = "ALL"

        stratification_used = "none"


    print(
        "Stratification:",
        stratification_used
    )


    # --------------------------------------------------------
    # Calculate group sizes
    # --------------------------------------------------------

    group_sizes = (
        evaluation_df["_stratum"]
        .value_counts()
        .sort_index()
    )

    # Proportional allocation before rounding
    raw_allocations = (
        group_sizes
        / len(evaluation_df)
        * SAMPLE_SIZE
    )


    # --------------------------------------------------------
    # Floor allocations
    #
    # IMPORTANT:
    # np.floor() is used instead of
    # raw_allocations.floor()
    # because raw_allocations is a Pandas Series.
    # --------------------------------------------------------

    allocations = (
        np.floor(raw_allocations)
        .astype(int)
    )


    # --------------------------------------------------------
    # Largest-remainder method
    #
    # Distribute rows lost during flooring.
    # --------------------------------------------------------

    remaining_slots = (
        SAMPLE_SIZE
        - allocations.sum()
    )

    remainders = (
        raw_allocations - allocations
    ).sort_values(
        ascending=False
    )


    for group in remainders.index:

        if remaining_slots <= 0:
            break

        # Do not allocate more rows than the group contains.
        if allocations.loc[group] < group_sizes.loc[group]:

            allocations.loc[group] += 1
            remaining_slots -= 1


    print("\nStratified allocations:")
    print(allocations)

    print(
        "\nTotal allocated after rounding:",
        allocations.sum()
    )


    # --------------------------------------------------------
    # Sample from each stratum
    # --------------------------------------------------------

    sampled_parts = []

    for group, n in allocations.items():

        if n <= 0:
            continue

        group_df = evaluation_df[
            evaluation_df["_stratum"] == group
        ]

        # Safety: never sample more rows than available.
        n = min(
            int(n),
            len(group_df)
        )

        if n == 0:
            continue

        sampled_group = group_df.sample(
            n=n,
            random_state=RANDOM_STATE
        )

        sampled_parts.append(
            sampled_group
        )


    # --------------------------------------------------------
    # Combine sampled groups
    # --------------------------------------------------------

    if sampled_parts:

        sample_df = pd.concat(
            sampled_parts,
            ignore_index=True
        )

    else:

        raise RuntimeError(
            "No rows were sampled. "
            "Check the stratification column."
        )


    # --------------------------------------------------------
    # Safety fill
    #
    # If group capacity or rounding caused fewer than
    # SAMPLE_SIZE rows, fill from unused eligible rows.
    # --------------------------------------------------------

    if len(sample_df) < SAMPLE_SIZE:

        selected_indices = set(
            sample_df["index"].astype(str)
        )

        remaining_candidates = evaluation_df[
            ~evaluation_df["index"].isin(
                selected_indices
            )
        ]

        extra_needed = (
            SAMPLE_SIZE - len(sample_df)
        )

        if len(remaining_candidates) < extra_needed:

            raise RuntimeError(
                "Not enough unused eligible rows to "
                "complete the requested sample."
            )

        extra = remaining_candidates.sample(
            n=extra_needed,
            random_state=RANDOM_STATE
        )

        sample_df = pd.concat(
            [
                sample_df,
                extra
            ],
            ignore_index=True
        )


    # --------------------------------------------------------
    # Safety trim
    # --------------------------------------------------------

    elif len(sample_df) > SAMPLE_SIZE:

        sample_df = sample_df.sample(
            n=SAMPLE_SIZE,
            random_state=RANDOM_STATE
        ).reset_index(drop=True)


    # --------------------------------------------------------
    # Remove helper column
    # --------------------------------------------------------

    sample_df = sample_df.drop(
        columns=["_stratum"],
        errors="ignore"
    )


    # --------------------------------------------------------
    # Shuffle final sample
    # --------------------------------------------------------

    sample_df = (
        sample_df
        .sample(
            frac=1,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Save fixed sample
    # --------------------------------------------------------

    sample_df.to_csv(
        SAMPLE_PATH,
        index=False
    )

    print(
        "\nNew fixed 300-row sample created."
    )


# ============================================================
# VERIFY SAMPLE
# ============================================================

sample_df["index"] = (
    sample_df["index"].astype(str)
)


# Exact sample-size check
assert len(sample_df) == SAMPLE_SIZE, (
    f"Expected {SAMPLE_SIZE} rows, "
    f"got {len(sample_df)}"
)


# No duplicate evaluation indices
assert (
    sample_df["index"].nunique()
    == SAMPLE_SIZE
), "Duplicate indices detected in sample."


# Required columns
required_columns = [
    "index",
    "input",
    "expected_answer",
    "model_response"
]

missing_columns = [
    col
    for col in required_columns
    if col not in sample_df.columns
]

if missing_columns:

    raise ValueError(
        f"Sample is missing required columns: "
        f"{missing_columns}"
    )


# ============================================================
# FINAL SAMPLE INFORMATION
# ============================================================

print()
print("=" * 80)
print("FIXED STRATIFIED SAMPLE")
print("=" * 80)

print("Rows:", len(sample_df))

print(
    "Unique indices:",
    sample_df["index"].nunique()
)

print(
    "Saved to:",
    SAMPLE_PATH
)

print("\nFirst 10 indices:")

print(
    sample_df["index"]
    .head(10)
    .tolist()
)

print("\nSample ready for calibration.")

✅ Groq client initialized
Judge model: openai/gpt-oss-120b

ELIGIBLE EVALUATION ROWS: 2998
Stratification: task_type

Stratified allocations:
_stratum
commonsense_reasoning     16
math_reasoning           273
science_reasoning         11
Name: count, dtype: int64

Total allocated after rounding: 300

New fixed 300-row sample created.

FIXED STRATIFIED SAMPLE
Rows: 300
Unique indices: 300
Saved to: /kaggle/working/llama_groq_final_300_sample.csv

First 10 indices:
['16319', '6404', '9897', '20065', '17935', '10430', '20086', '22988', '8137', '27275']

Sample ready for calibration.


In [36]:
# ============================================================
# CELL 2 — GROQ JUDGE FUNCTION
# GPT-OSS-120B
# ============================================================

import json
import re
import time


# ============================================================
# JUDGE PROMPT
# ============================================================

def build_judge_prompt(row):

    question = str(row["input"])
    reference_answer = str(row["expected_answer"])
    llama_response = str(row["model_response"])

    prompt = f"""
You are an expert evaluator judging the response of a fine-tuned Llama model.

Your task is to independently evaluate:

1. Whether the QUESTION is valid and solvable.
2. Whether the REFERENCE ANSWER is correct for that question.
3. Whether the Llama's FINAL ANSWER is correct.
4. Whether the Llama's REASONING is logically and mathematically valid.

IMPORTANT:

Do NOT judge the Llama answer merely by comparing text strings.

You must understand the question and independently determine the correct result.

Also do NOT assume that the Llama reasoning is correct just because its final answer is correct.

A correct final answer with invalid reasoning must receive:

"answer_correct": true
"reasoning_correct": false

Likewise, an incorrect final answer with otherwise meaningful reasoning must receive:

"answer_correct": false

You must evaluate the reasoning itself.

IMPORTANT ORDER:

First analyze the problem and the Llama response carefully.

Then produce your final structured judgment.

Do not guess the verdict before analyzing the response.

------------------------------------------------------------
QUESTION
------------------------------------------------------------

{question}

------------------------------------------------------------
REFERENCE ANSWER
------------------------------------------------------------

{reference_answer}

------------------------------------------------------------
LLAMA RESPONSE
------------------------------------------------------------

{llama_response}

------------------------------------------------------------
EVALUATION
------------------------------------------------------------

Determine:

- Is the question valid?
- Is the reference answer correct?
- Is the Llama final answer correct?
- Is the Llama reasoning correct?

If the question is invalid or the reference answer is wrong, explain that clearly.

For the reasoning judgment, check whether the actual steps used by Llama are logically valid.
Do not punish Llama merely for being brief.
Do not require the exact wording of the reference solution.
Equivalent mathematical reasoning is acceptable.

Return ONLY one JSON object after completing your analysis.

The JSON must contain exactly these fields:

{{
  "question_valid": true,
  "reference_answer_correct": true,
  "answer_correct": true,
  "reasoning_correct": true,
  "judge_evidence": "brief explanation supporting the verdicts"
}}

The values for the first four fields must be true or false.

"judge_evidence" must be a concise explanation of the actual evidence used to reach the verdicts.

Do not put markdown around the JSON.
"""


    return prompt


# ============================================================
# JSON EXTRACTION
# ============================================================

def extract_json_object(text):

    if text is None:
        return None

    text = str(text).strip()

    # --------------------------------------------------------
    # First try the entire response
    # --------------------------------------------------------

    try:
        parsed = json.loads(text)

        if isinstance(parsed, dict):
            return parsed

    except Exception:
        pass


    # --------------------------------------------------------
    # Try extracting the last JSON object
    # --------------------------------------------------------

    matches = re.findall(
        r'\{(?:[^{}]|(?:\{[^{}]*\}))*\}',
        text,
        flags=re.DOTALL
    )

    for candidate in reversed(matches):

        try:

            parsed = json.loads(candidate)

            if isinstance(parsed, dict):
                return parsed

        except Exception:
            continue


    return None


# ============================================================
# SINGLE-ROW JUDGE
# ============================================================

def judge_one_row(
    row,
    reasoning_effort="high",
    max_retries=3,
    initial_retry_wait=5
):

    prompt = build_judge_prompt(row)

    last_error = None

    for attempt in range(max_retries):

        start_time = time.time()

        try:

            response = client.chat.completions.create(

                model=GROQ_MODEL,

                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a rigorous LLM evaluator. "
                            "Analyze the response before producing "
                            "your final JSON judgment."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],

                # IMPORTANT:
                # We intentionally use HIGH for calibration.
                reasoning_effort=reasoning_effort,

                # Do NOT use:
                # reasoning_format="hidden"

                max_completion_tokens=MAX_COMPLETION_TOKENS,

                temperature=0
            )


            # ------------------------------------------------
            # Extract response
            # ------------------------------------------------

            raw_content = (
                response
                .choices[0]
                .message
                .content
            )

            elapsed = time.time() - start_time


            # ------------------------------------------------
            # Parse JSON
            # ------------------------------------------------

            parsed = extract_json_object(
                raw_content
            )


            if parsed is None:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "FAILED",
                    "raw_output": raw_content,
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Validate required fields
            # ------------------------------------------------

            required_fields = [
                "question_valid",
                "reference_answer_correct",
                "answer_correct",
                "reasoning_correct",
                "judge_evidence"
            ]

            missing_fields = [
                field
                for field in required_fields
                if field not in parsed
            ]


            if missing_fields:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": (
                        f"Missing fields: {missing_fields}"
                    ),
                    "parse_status": "INVALID_SCHEMA",
                    "raw_output": raw_content,
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Normalize boolean fields
            # ------------------------------------------------

            boolean_fields = [
                "question_valid",
                "reference_answer_correct",
                "answer_correct",
                "reasoning_correct"
            ]

            for field in boolean_fields:

                value = parsed[field]

                if isinstance(value, str):

                    value_lower = value.lower().strip()

                    if value_lower == "true":
                        parsed[field] = True

                    elif value_lower == "false":
                        parsed[field] = False

                    else:
                        parsed[field] = None


            # ------------------------------------------------
            # Return successful result
            # ------------------------------------------------

            return {
                "question_valid": parsed[
                    "question_valid"
                ],

                "reference_answer_correct": parsed[
                    "reference_answer_correct"
                ],

                "answer_correct": parsed[
                    "answer_correct"
                ],

                "reasoning_correct": parsed[
                    "reasoning_correct"
                ],

                "judge_evidence": str(
                    parsed["judge_evidence"]
                ),

                "parse_status": "OK",

                "raw_output": raw_content,

                "time_seconds": round(
                    elapsed,
                    2
                )
            }


        except Exception as e:

            last_error = str(e)

            # Retry with exponential backoff
            if attempt < max_retries - 1:

                wait_time = (
                    initial_retry_wait
                    * (2 ** attempt)
                )

                print(
                    f"Retry {attempt + 1}/{max_retries - 1} "
                    f"after error: {last_error}"
                )

                time.sleep(wait_time)


    # ========================================================
    # ALL RETRIES FAILED
    # ========================================================

    return {
        "question_valid": None,
        "reference_answer_correct": None,
        "answer_correct": None,
        "reasoning_correct": None,
        "judge_evidence": None,
        "parse_status": "ERROR",
        "raw_output": "",
        "error": last_error,
        "time_seconds": None
    }


# ============================================================
# VERIFY FUNCTION EXISTS
# ============================================================

print("=" * 80)
print("JUDGE FUNCTION READY")
print("=" * 80)

print("Model:", GROQ_MODEL)
print("Function:", judge_one_row.__name__)
print("Calibration reasoning:", CALIBRATION_REASONING_EFFORT)
print("Full evaluation reasoning:", FULL_REASONING_EFFORT)
print("Max completion tokens:", MAX_COMPLETION_TOKENS)

print("\n✅ Cell 2 loaded successfully.")

JUDGE FUNCTION READY
Model: openai/gpt-oss-120b
Function: judge_one_row
Calibration reasoning: high
Full evaluation reasoning: medium
Max completion tokens: 1200

✅ Cell 2 loaded successfully.


In [38]:
# ============================================================
# CELL 3 — 10-ROW HIGH-REASONING CALIBRATION
# ============================================================

import pandas as pd
import time


print("=" * 80)
print("10-ROW HIGH-REASONING CALIBRATION")
print("=" * 80)


# ------------------------------------------------------------
# Select fixed 10 rows from the existing 300-row sample
# ------------------------------------------------------------

calibration_df = (
    sample_df
    .sample(
        n=CALIBRATION_SIZE,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)


print(
    "Indices:",
    calibration_df["index"].tolist()
)


# ------------------------------------------------------------
# Run judge
# ------------------------------------------------------------

calibration_results = []


for i, (_, row) in enumerate(
    calibration_df.iterrows(),
    start=1
):

    print()
    print("-" * 80)

    start = time.time()

    result = judge_one_row(
        row,
        reasoning_effort=CALIBRATION_REASONING_EFFORT,
        max_retries=MAX_RETRIES,
        initial_retry_wait=INITIAL_RETRY_WAIT
    )

    elapsed = time.time() - start


    # --------------------------------------------------------
    # Add row information HERE
    # --------------------------------------------------------

    result["index"] = str(
        row["index"]
    )

    result["expected_answer"] = str(
        row["expected_answer"]
    )

    result["input"] = str(
        row["input"]
    )

    result["model_response"] = str(
        row["model_response"]
    )


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    calibration_results.append(
        result
    )


    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    print(
        f"Completed {i}/{CALIBRATION_SIZE} | "
        f"Index: {result['index']}"
    )

    print(
        "Question valid:",
        result["question_valid"]
    )

    print(
        "Reference correct:",
        result["reference_answer_correct"]
    )

    print(
        "Llama answer correct:",
        result["answer_correct"]
    )

    print(
        "Llama reasoning correct:",
        result["reasoning_correct"]
    )

    print(
        "Parse:",
        result["parse_status"]
    )

    print(
        "Time:",
        result.get(
            "time_seconds",
            round(elapsed, 2)
        ),
        "seconds"
    )

    if result.get("judge_evidence"):

        print(
            "Judge evidence:",
            result["judge_evidence"]
        )

    if result.get("raw_output"):

        print(
            "Raw output:",
            result["raw_output"]
        )


# ============================================================
# CREATE CALIBRATION DATAFRAME
# ============================================================

calibration_results_df = pd.DataFrame(
    calibration_results
)


# ------------------------------------------------------------
# Save calibration
# ------------------------------------------------------------

calibration_results_df.to_csv(
    CALIBRATION_PATH,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 80)
print("CALIBRATION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    len(calibration_results_df)
)

print(
    "Saved to:",
    CALIBRATION_PATH
)


print("\nParse status:")
print(
    calibration_results_df[
        "parse_status"
    ].value_counts(dropna=False)
)


print("\nAnswer correctness:")
print(
    calibration_results_df[
        "answer_correct"
    ].value_counts(dropna=False)
)


print("\nReasoning correctness:")
print(
    calibration_results_df[
        "reasoning_correct"
    ].value_counts(dropna=False)
)

10-ROW HIGH-REASONING CALIBRATION
Indices: ['15973', '23498', '5464', '27275', '11204', '15014', '279', '18488', '10430', '17816']

--------------------------------------------------------------------------------
Completed 1/10 | Index: 15973
Question valid: True
Reference correct: True
Llama answer correct: False
Llama reasoning correct: False
Parse: OK
Time: 2.69 seconds
Judge evidence: The problem asks for total cost in silvers: 5 spellbooks at 5 gold each (25 gold), 1 owl at 28 gold (total 53 gold), and 3 potion kits at 20 silver each (60 silver). Converting 53 gold to silver at 9 silver per gold gives 477 silver; adding 60 silver yields 537 silver. The reference answer matches this. The Llama answer of 1087 silver is incorrect. Its reasoning mistakenly adds the 60 silver to the gold total, treats the sum as gold, uses an incorrect conversion (113*9 incorrectly computed as 1027), and double‑counts the potion kit silver, so the reasoning is invalid.
Raw output: {
  "question_valid":

In [39]:
# ============================================================
# CELL 4 — RETRY ONLY FAILED CALIBRATION ROWS
# ============================================================

FAILED_INDICES = ["27275", "11204"]

retry_rows = sample_df[
    sample_df["index"].astype(str).isin(FAILED_INDICES)
].copy()

print("=" * 80)
print("RETRYING FAILED CALIBRATION ROWS")
print("=" * 80)

print("Rows:", len(retry_rows))
print("Indices:", retry_rows["index"].astype(str).tolist())


retry_results = []

for i, (_, row) in enumerate(
    retry_rows.iterrows(),
    start=1
):

    print()
    print("-" * 80)

    print(
        f"Retry {i}/{len(retry_rows)} | "
        f"Index: {row['index']}"
    )

    result = judge_one_row(
        row,
        reasoning_effort=CALIBRATION_REASONING_EFFORT,
        max_retries=1
    )

    # Add row information
    result["index"] = str(row["index"])

    result["expected_answer"] = str(
        row["expected_answer"]
    )

    result["input"] = str(
        row["input"]
    )

    result["model_response"] = str(
        row["model_response"]
    )

    retry_results.append(result)

    print(
        "Question valid:",
        result["question_valid"]
    )

    print(
        "Reference correct:",
        result["reference_answer_correct"]
    )

    print(
        "Llama answer correct:",
        result["answer_correct"]
    )

    print(
        "Llama reasoning correct:",
        result["reasoning_correct"]
    )

    print(
        "Parse:",
        result["parse_status"]
    )

    print(
        "Time:",
        result.get("time_seconds"),
        "seconds"
    )

    if result.get("judge_evidence"):
        print(
            "Judge evidence:",
            result["judge_evidence"]
        )

    if result.get("raw_output"):
        print(
            "Raw output:",
            result["raw_output"]
        )


# ============================================================
# SAVE RETRY RESULTS
# ============================================================

retry_df = pd.DataFrame(retry_results)

RETRY_PATH = (
    "/kaggle/working/"
    "llama_groq_calibration_retry_2.csv"
)

retry_df.to_csv(
    RETRY_PATH,
    index=False
)


print()
print("=" * 80)
print("RETRY COMPLETE")
print("=" * 80)

print("Rows:", len(retry_df))

print("\nParse status:")
print(
    retry_df["parse_status"]
    .value_counts(dropna=False)
)

print("\nSaved to:")
print(RETRY_PATH)

RETRYING FAILED CALIBRATION ROWS
Rows: 2
Indices: ['27275', '11204']

--------------------------------------------------------------------------------
Retry 1/2 | Index: 27275
Question valid: None
Reference correct: None
Llama answer correct: None
Llama reasoning correct: None
Parse: FAILED
Time: 3.07 seconds

--------------------------------------------------------------------------------
Retry 2/2 | Index: 11204
Question valid: None
Reference correct: None
Llama answer correct: None
Llama reasoning correct: None
Parse: FAILED
Time: 4.56 seconds

RETRY COMPLETE
Rows: 2

Parse status:
parse_status
FAILED    2
Name: count, dtype: int64

Saved to:
/kaggle/working/llama_groq_calibration_retry_2.csv


In [40]:
# ============================================================
# DIAGNOSE THE 2 FAILED RETRIES
# ============================================================

print("FAILED RETRY DETAILS")
print("=" * 80)

for _, row in retry_df.iterrows():

    print("\nIndex:", row["index"])
    print("Parse status:", row["parse_status"])
    print("Error:", row.get("error", "NO ERROR FIELD"))
    print("Raw output:", repr(row.get("raw_output", "")))
    print("Time:", row.get("time_seconds"))

FAILED RETRY DETAILS

Index: 27275
Parse status: FAILED
Error: NO ERROR FIELD
Raw output: ''
Time: 3.07

Index: 11204
Parse status: FAILED
Error: NO ERROR FIELD
Raw output: ''
Time: 4.56


In [41]:
# ============================================================
# GROQ RATE-LIMIT TEST
# ============================================================

import time

print("=" * 80)
print("TESTING GROQ GPT-OSS-120B AVAILABILITY")
print("=" * 80)

try:
    start = time.time()

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": "Reply with exactly: OK"
            }
        ],
        reasoning_effort="low",
        max_completion_tokens=10
    )

    elapsed = time.time() - start

    content = response.choices[0].message.content

    print("✅ REQUEST SUCCESSFUL")
    print("Response:", repr(content))
    print("Time:", round(elapsed, 2), "seconds")

    if hasattr(response, "usage"):
        print("\nUsage:")
        print(response.usage)

except Exception as e:

    print("❌ REQUEST FAILED")
    print("Error type:", type(e).__name__)
    print("Error:")
    print(str(e))

TESTING GROQ GPT-OSS-120B AVAILABILITY
✅ REQUEST SUCCESSFUL
Response: ''
Time: 0.47 seconds

Usage:
CompletionUsage(completion_tokens=10, prompt_tokens=76, total_tokens=86, completion_time=0.020926316, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=8), prompt_time=0.002870948, prompt_tokens_details=None, queue_time=0.286998769, total_time=0.023797264)


In [42]:
# ============================================================
# TEST ONE FAILED CALIBRATION ROW
# ACTUAL JUDGE CONFIGURATION
# ============================================================

test_index = "11204"

test_row = sample_df[
    sample_df["index"].astype(str) == test_index
].iloc[0]

print("=" * 80)
print("TESTING ACTUAL JUDGE")
print("=" * 80)
print("Index:", test_index)
print("Reasoning effort:", CALIBRATION_REASONING_EFFORT)

result_11204 = judge_one_row(
    test_row,
    reasoning_effort=CALIBRATION_REASONING_EFFORT,
    max_retries=1
)

print("\nQuestion valid:", result_11204["question_valid"])
print(
    "Reference correct:",
    result_11204["reference_answer_correct"]
)
print(
    "Llama answer correct:",
    result_11204["answer_correct"]
)
print(
    "Llama reasoning correct:",
    result_11204["reasoning_correct"]
)
print(
    "Parse:",
    result_11204["parse_status"]
)
print(
    "Time:",
    result_11204.get("time_seconds"),
    "seconds"
)

print("\nJudge evidence:")
print(result_11204.get("judge_evidence"))

print("\nRaw output:")
print(result_11204.get("raw_output"))

TESTING ACTUAL JUDGE
Index: 11204
Reasoning effort: high

Question valid: None
Reference correct: None
Llama answer correct: None
Llama reasoning correct: None
Parse: FAILED
Time: 3.0 seconds

Judge evidence:
None

Raw output:



In [43]:
# ============================================================
# DIRECT GPT-OSS-120B DIAGNOSTIC
# ============================================================

row = sample_df[
    sample_df["index"].astype(str) == "11204"
].iloc[0]

prompt = build_judge_prompt(row)

print("=" * 80)
print("DIRECT GPT-OSS-120B DIAGNOSTIC")
print("=" * 80)

try:

    start = time.time()

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",

        messages=[
            {
                "role": "system",
                "content": (
                    "You are a rigorous evaluator. "
                    "Analyze the problem before giving "
                    "your final JSON judgment."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        reasoning_effort="high",

        max_completion_tokens=2000,

        temperature=0
    )

    elapsed = time.time() - start

    print("\nREQUEST SUCCESSFUL")
    print("Time:", round(elapsed, 2), "seconds")

    print("\nRAW RESPONSE OBJECT:")
    print(response)

    print("\nMESSAGE CONTENT:")
    print(
        repr(
            response.choices[0].message.content
        )
    )

    print("\nFINISH REASON:")
    print(
        response.choices[0].finish_reason
    )

    print("\nUSAGE:")
    print(response.usage)

except Exception as e:

    print("\nREQUEST FAILED")
    print("Exception:", type(e).__name__)
    print(str(e))

DIRECT GPT-OSS-120B DIAGNOSTIC

REQUEST SUCCESSFUL
Time: 4.67 seconds

RAW RESPONSE OBJECT:
ChatCompletion(id='chatcmpl-0253bd48-6eea-4e95-b742-f87ab70c3d40', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to evaluate the problem: "In 1 hour, a boat goes 6 km along the stream and 12 km against the stream. The speed of the boat in still water (in km/hr) is : Choices: A. 3 km/hr. B. 2 km/hr. C. 6 km/hr. D. 8 km/hr. E. 9 km/hr."\n\nInterpretation: The boat travels 6 km downstream (with the stream) and 12 km upstream (against the stream) in total time of 1 hour? Or "In 1 hour, a boat goes 6 km along the stream and 12 km against the stream." Means that in one hour, the boat travels 6 km downstream and 12 km upstream. So total distance traveled is 18 km in 1 hour, but the speeds differ for each direction. Let v_b = speed in still water, v_

In [53]:
# ============================================================
# GPT-OSS-120B JUDGE V3
# HIGH REASONING + SIMPLE JSON OUTPUT
# ============================================================

def judge_one_row(
    row,
    reasoning_effort="high",
    max_retries=2,
    initial_retry_wait=15
):

    question = str(row["input"])
    reference_answer = str(row["expected_answer"])
    llama_response = str(row["model_response"])

    prompt = f"""
You are evaluating a fine-tuned Llama language model.

Carefully solve the question yourself first. Then evaluate the
Llama response.

QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

LLAMA RESPONSE:
{llama_response}

Evaluate these four things:

1. question_valid
   Is the question valid and sufficiently specified?

2. reference_answer_correct
   Is the reference answer actually correct?

3. answer_correct
   Is Llama's final answer correct?

4. reasoning_correct
   Is Llama's reasoning logically or mathematically valid?

Rules:

- Do NOT use string matching alone.
- Independently solve the question.
- A correct final answer does NOT imply correct reasoning.
- If the final answer is correct but the reasoning is invalid:
  answer_correct = true
  reasoning_correct = false
- If Llama provides no reasoning where reasoning is required:
  reasoning_correct = false.
- Equivalent valid reasoning is acceptable.
- If the question itself is invalid, reflect that in question_valid.
- If the reference answer is wrong, reflect that in
  reference_answer_correct.

Think through the problem carefully before producing the verdict.

At the very end, output ONLY this JSON object:

{{
  "question_valid": true,
  "reference_answer_correct": true,
  "answer_correct": true,
  "reasoning_correct": true
}}

Do not output anything before or after the JSON.
"""


    for attempt in range(max_retries):

        start_time = time.time()

        try:

            response = client.chat.completions.create(

                model=GROQ_MODEL,

                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],

                reasoning_effort=reasoning_effort,

                # Give the reasoning model enough room.
                max_completion_tokens=5000,

                temperature=0
            )


            elapsed = time.time() - start_time

            message = response.choices[0].message

            raw_content = (
                message.content or ""
            ).strip()

            finish_reason = (
                response.choices[0].finish_reason
            )


            # ------------------------------------------------
            # Empty output
            # ------------------------------------------------

            if not raw_content:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "EMPTY_RESPONSE",
                    "raw_output": raw_content,
                    "finish_reason": finish_reason,
                    "error": (
                        "Model finished without visible JSON."
                    ),
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Extract JSON
            # ------------------------------------------------

            parsed = extract_json_object(
                raw_content
            )


            if parsed is None:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "PARSE_FAILED",
                    "raw_output": raw_content,
                    "finish_reason": finish_reason,
                    "error": (
                        "Could not extract JSON "
                        "from model output."
                    ),
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Validate fields
            # ------------------------------------------------

            required_fields = [
                "question_valid",
                "reference_answer_correct",
                "answer_correct",
                "reasoning_correct"
            ]

            missing = [
                field
                for field in required_fields
                if field not in parsed
            ]

            if missing:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "INVALID_SCHEMA",
                    "raw_output": raw_content,
                    "finish_reason": finish_reason,
                    "error": (
                        f"Missing fields: {missing}"
                    ),
                    "time_seconds": round(
                        elapsed,
                        2
                    )
                }


            # ------------------------------------------------
            # Successful judgment
            # ------------------------------------------------

            return {
                "question_valid": bool(
                    parsed["question_valid"]
                ),

                "reference_answer_correct": bool(
                    parsed[
                        "reference_answer_correct"
                    ]
                ),

                "answer_correct": bool(
                    parsed["answer_correct"]
                ),

                "reasoning_correct": bool(
                    parsed["reasoning_correct"]
                ),

                # Evidence is intentionally empty here.
                # We can audit the raw reasoning separately.
                "judge_evidence": None,

                "parse_status": "OK",

                "raw_output": raw_content,

                "finish_reason": finish_reason,

                "time_seconds": round(
                    elapsed,
                    2
                )
            }


        except Exception as e:

            error_text = str(e)

            print(
                f"Attempt {attempt + 1}/{max_retries} failed:"
            )
            print(error_text)


            if attempt < max_retries - 1:

                wait_time = (
                    initial_retry_wait
                    * (2 ** attempt)
                )

                print(
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                return {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "API_ERROR",
                    "raw_output": "",
                    "finish_reason": None,
                    "error": error_text,
                    "time_seconds": None
                }

In [54]:
test_index = "11204"

test_row = sample_df[
    sample_df["index"].astype(str) == test_index
].iloc[0]

result_11204 = judge_one_row(
    test_row,
    reasoning_effort="high",
    max_retries=1
)

print("=" * 80)
print("11204 TEST")
print("=" * 80)

print("Question valid:", result_11204["question_valid"])
print(
    "Reference correct:",
    result_11204["reference_answer_correct"]
)
print(
    "Llama answer correct:",
    result_11204["answer_correct"]
)
print(
    "Llama reasoning correct:",
    result_11204["reasoning_correct"]
)

print(
    "Parse:",
    result_11204["parse_status"]
)

print(
    "Finish reason:",
    result_11204.get("finish_reason")
)

print(
    "Time:",
    result_11204.get("time_seconds")
)

print(
    "\nRaw JSON:"
)

print(
    result_11204["raw_output"]
)

if result_11204.get("error"):
    print("\nERROR:")
    print(result_11204["error"])

11204 TEST
Question valid: True
Reference correct: True
Llama answer correct: True
Llama reasoning correct: False
Parse: OK
Finish reason: stop
Time: 4.38

Raw JSON:
{
  "question_valid": true,
  "reference_answer_correct": true,
  "answer_correct": true,
  "reasoning_correct": false
}


In [55]:
test_index = "27275"

test_row = sample_df[
    sample_df["index"].astype(str) == test_index
].iloc[0]

result_27275 = judge_one_row(
    test_row,
    reasoning_effort="high",
    max_retries=1
)

print("=" * 80)
print("27275 TEST")
print("=" * 80)

print("Question valid:", result_27275["question_valid"])
print(
    "Reference correct:",
    result_27275["reference_answer_correct"]
)
print(
    "Llama answer correct:",
    result_27275["answer_correct"]
)
print(
    "Llama reasoning correct:",
    result_27275["reasoning_correct"]
)

print(
    "Parse:",
    result_27275["parse_status"]
)

print(
    "Finish reason:",
    result_27275.get("finish_reason")
)

print(
    "Time:",
    result_27275.get("time_seconds")
)

print("\nRaw JSON:")
print(result_27275["raw_output"])

if result_27275.get("error"):
    print("\nERROR:")
    print(result_27275["error"])

27275 TEST
Question valid: True
Reference correct: True
Llama answer correct: True
Llama reasoning correct: False
Parse: OK
Finish reason: stop
Time: 3.93

Raw JSON:
{
  "question_valid": true,
  "reference_answer_correct": true,
  "answer_correct": true,
  "reasoning_correct": false
}


In [48]:
# ============================================================
# SHOW THE ACTUAL API ERROR FROM 11204
# ============================================================

print("=" * 80)
print("11204 API ERROR")
print("=" * 80)

print("Error:")
print(result_11204.get("error"))

print("\nParse status:")
print(result_11204.get("parse_status"))

print("\nFinish reason:")
print(result_11204.get("finish_reason"))

11204 API ERROR
Error:
Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}

Parse status:
API_ERROR

Finish reason:
None


In [57]:
# ============================================================
# BUILD FINAL 10-ROW CALIBRATION
# USING THE TWO SUCCESSFUL RETRIES ALREADY IN MEMORY
# ============================================================

# Create retry_success directly from the two results
retry_success = pd.DataFrame([
    {
        "index": "11204",
        **result_11204
    },
    {
        "index": "27275",
        **result_27275
    }
])

# Keep only successful judgments
retry_success = retry_success[
    retry_success["parse_status"] == "OK"
].copy()

# Make index type consistent
retry_success["index"] = (
    retry_success["index"].astype(str)
)

print("=" * 80)
print("SUCCESSFUL RETRIES")
print("=" * 80)

print("Rows:", len(retry_success))
print("Indices:", retry_success["index"].tolist())


# ============================================================
# ORIGINAL 8 SUCCESSFUL CALIBRATION ROWS
# ============================================================

original_success = calibration_results_df[
    calibration_results_df["parse_status"] == "OK"
].copy()

original_success["index"] = (
    original_success["index"].astype(str)
)


# ============================================================
# COMBINE
# ============================================================

final_calibration_df = pd.concat(
    [
        original_success,
        retry_success
    ],
    ignore_index=True
)


# Remove duplicate indices if any
final_calibration_df = (
    final_calibration_df
    .drop_duplicates(
        subset=["index"],
        keep="last"
    )
    .reset_index(drop=True)
)


# ============================================================
# VERIFY
# ============================================================

print()
print("=" * 80)
print("FINAL CALIBRATION DATASET")
print("=" * 80)

print("Rows:", len(final_calibration_df))

print(
    "Unique indices:",
    final_calibration_df["index"].nunique()
)

print("\nIndices:")
print(
    final_calibration_df["index"].tolist()
)


print("\nQuestion validity:")
print(
    final_calibration_df[
        "question_valid"
    ].value_counts(dropna=False)
)


print("\nReference answer validity:")
print(
    final_calibration_df[
        "reference_answer_correct"
    ].value_counts(dropna=False)
)


print("\nLlama answer correctness:")
print(
    final_calibration_df[
        "answer_correct"
    ].value_counts(dropna=False)
)


print("\nLlama reasoning correctness:")
print(
    final_calibration_df[
        "reasoning_correct"
    ].value_counts(dropna=False)
)


print("\nParse status:")
print(
    final_calibration_df[
        "parse_status"
    ].value_counts(dropna=False)
)


# ============================================================
# SAVE FINAL CALIBRATION
# ============================================================

CALIBRATION_FINAL_PATH = (
    "/kaggle/working/"
    "llama_groq_final_calibration_10.csv"
)

final_calibration_df.to_csv(
    CALIBRATION_FINAL_PATH,
    index=False
)

print()
print("=" * 80)
print("CALIBRATION SAVED")
print("=" * 80)

print(CALIBRATION_FINAL_PATH)

SUCCESSFUL RETRIES
Rows: 2
Indices: ['11204', '27275']

FINAL CALIBRATION DATASET
Rows: 10
Unique indices: 10

Indices:
['15973', '23498', '5464', '15014', '279', '18488', '10430', '17816', '11204', '27275']

Question validity:
question_valid
True    10
Name: count, dtype: int64

Reference answer validity:
reference_answer_correct
True    10
Name: count, dtype: int64

Llama answer correctness:
answer_correct
True     6
False    4
Name: count, dtype: int64

Llama reasoning correctness:
reasoning_correct
False    8
True     2
Name: count, dtype: int64

Parse status:
parse_status
OK    10
Name: count, dtype: int64

CALIBRATION SAVED
/kaggle/working/llama_groq_final_calibration_10.csv


In [63]:
# ============================================================
# RESUMABLE 300-ROW EVALUATION
# SAVE AFTER EVERY SUCCESSFUL ROW
# STOP IMMEDIATELY ON RATE LIMIT
# ============================================================

import os
import time
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

EVAL_SIZE = 300

REASONING_EFFORT = "high"

MAX_RETRIES = 2
RETRY_WAIT = 20

RANDOM_SAMPLE_PATH = (
    "/kaggle/working/"
    "llama_groq_random_300_sample.csv"
)

CHECKPOINT_PATH = (
    "/kaggle/working/"
    "llama_groq_random_300_checkpoint.csv"
)

FINAL_RESULTS_PATH = (
    "/kaggle/working/"
    "llama_groq_random_300_results.csv"
)

# ============================================================
# 1. LOAD THE EXISTING FIXED 300-ROW SAMPLE
# ============================================================

if not os.path.exists(RANDOM_SAMPLE_PATH):

    raise FileNotFoundError(
        "The fixed 300-row sample was not found:\n"
        f"{RANDOM_SAMPLE_PATH}\n\n"
        "Do NOT create a new sample here."
    )

sample_df = pd.read_csv(
    RANDOM_SAMPLE_PATH
)

sample_df["index"] = (
    sample_df["index"].astype(str)
)

# ============================================================
# VERIFY SAMPLE
# ============================================================

if len(sample_df) != EVAL_SIZE:

    raise ValueError(
        f"Expected {EVAL_SIZE} rows, "
        f"but found {len(sample_df)}."
    )

if sample_df["index"].nunique() != EVAL_SIZE:

    raise ValueError(
        "Duplicate indices detected in "
        "the fixed 300-row sample."
    )

print("=" * 80)
print("FIXED 300-ROW SAMPLE")
print("=" * 80)

print("Rows:", len(sample_df))
print(
    "Unique indices:",
    sample_df["index"].nunique()
)

# ============================================================
# 2. LOAD EXISTING CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT_PATH):

    results_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    results_df["index"] = (
        results_df["index"].astype(str)
    )

    print()
    print("=" * 80)
    print("EXISTING CHECKPOINT FOUND")
    print("=" * 80)

    print(
        "Completed rows in checkpoint:",
        len(results_df)
    )

else:

    results_df = pd.DataFrame()

    print()
    print("=" * 80)
    print("NO CHECKPOINT FOUND")
    print("=" * 80)

    print(
        "Starting evaluation from the beginning."
    )

# ============================================================
# 3. REMOVE ANY DUPLICATES FROM CHECKPOINT
# ============================================================

if not results_df.empty:

    results_df = (
        results_df
        .drop_duplicates(
            subset=["index"],
            keep="last"
        )
        .reset_index(drop=True)
    )

# ============================================================
# 4. DETERMINE COMPLETED ROWS
# ============================================================

if results_df.empty:

    completed_indices = set()

else:

    completed_indices = set(
        results_df["index"]
        .astype(str)
    )

# Only evaluate rows that aren't already completed

remaining_df = sample_df[
    ~sample_df["index"].isin(
        completed_indices
    )
].copy()

print()
print("=" * 80)
print("RESUME STATUS")
print("=" * 80)

print(
    "Total sample:",
    len(sample_df)
)

print(
    "Already completed:",
    len(completed_indices)
)

print(
    "Remaining:",
    len(remaining_df)
)

# ============================================================
# 5. EVALUATION LOOP
# ============================================================

rate_limit_hit = False

for position, (_, row) in enumerate(
    remaining_df.iterrows(),
    start=1
):

    index = str(row["index"])

    overall_number = (
        len(completed_indices)
        + position
    )

    print()
    print("=" * 80)

    print(
        f"Evaluating "
        f"{overall_number}/{EVAL_SIZE}"
    )

    print(
        f"Index: {index}"
    )

    print("=" * 80)

    result = None

    # --------------------------------------------------------
    # RETRY LOOP
    # --------------------------------------------------------

    for attempt in range(
        MAX_RETRIES
    ):

        try:

            result = judge_one_row(
                row,
                reasoning_effort=(
                    REASONING_EFFORT
                ),
                max_retries=1
            )

            parse_status = (
                result.get(
                    "parse_status"
                )
            )

            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            if parse_status == "OK":

                break

            # ------------------------------------------------
            # API ERROR
            # ------------------------------------------------

            if parse_status == "API_ERROR":

                error_text = str(
                    result.get(
                        "error",
                        ""
                    )
                )

                # --------------------------------------------
                # RATE LIMIT DETECTION
                # --------------------------------------------

                if (
                    "429" in error_text
                    or
                    "rate limit" in
                    error_text.lower()
                    or
                    "rate_limit_exceeded"
                    in error_text.lower()
                ):

                    print()
                    print(
                        "🚨 GROQ RATE LIMIT "
                        "DETECTED"
                    )

                    print(
                        error_text
                    )

                    rate_limit_hit = True

                    break

            # ------------------------------------------------
            # NON-RATE-LIMIT FAILURE
            # ------------------------------------------------

            if attempt < MAX_RETRIES - 1:

                print(
                    f"Retrying in "
                    f"{RETRY_WAIT} seconds..."
                )

                time.sleep(
                    RETRY_WAIT
                )

        except Exception as e:

            error_text = str(e)

            print()
            print(
                "Exception:"
            )

            print(
                error_text
            )

            # --------------------------------------------
            # RATE LIMIT DETECTION
            # --------------------------------------------

            if (
                "429" in error_text
                or
                "rate limit" in
                error_text.lower()
                or
                "rate_limit_exceeded"
                in error_text.lower()
            ):

                print()
                print(
                    "🚨 GROQ RATE LIMIT "
                    "DETECTED"
                )

                rate_limit_hit = True

                result = {
                    "question_valid": None,
                    "reference_answer_correct": None,
                    "answer_correct": None,
                    "reasoning_correct": None,
                    "judge_evidence": None,
                    "parse_status": "API_ERROR",
                    "raw_output": "",
                    "finish_reason": None,
                    "error": error_text,
                    "time_seconds": None
                }

                break

            # --------------------------------------------
            # OTHER ERROR
            # --------------------------------------------

            result = {
                "question_valid": None,
                "reference_answer_correct": None,
                "answer_correct": None,
                "reasoning_correct": None,
                "judge_evidence": None,
                "parse_status": "API_ERROR",
                "raw_output": "",
                "finish_reason": None,
                "error": error_text,
                "time_seconds": None
            }

            if attempt < MAX_RETRIES - 1:

                print(
                    f"Retrying in "
                    f"{RETRY_WAIT} seconds..."
                )

                time.sleep(
                    RETRY_WAIT
                )

    # ========================================================
    # STOP IMMEDIATELY IF RATE LIMITED
    # ========================================================

    if rate_limit_hit:

        print()
        print("=" * 80)
        print(
            "STOPPING EVALUATION"
        )
        print("=" * 80)

        print(
            "Reason: Groq token limit "
            "was reached."
        )

        print(
            "The completed rows have "
            "already been saved."
        )

        break

    # ========================================================
    # 6. BUILD RESULT ROW
    # ========================================================

    result_row = {
        "index": index,

        "input": row["input"],

        "expected_answer": (
            row["expected_answer"]
        ),

        "model_response": (
            row["model_response"]
        ),

        "dataset": (
            row["dataset"]
            if "dataset" in row.index
            else None
        ),

        "task_type": (
            row["task_type"]
            if "task_type" in row.index
            else None
        ),

        "question_valid": (
            result.get(
                "question_valid"
            )
        ),

        "reference_answer_correct": (
            result.get(
                "reference_answer_correct"
            )
        ),

        "answer_correct": (
            result.get(
                "answer_correct"
            )
        ),

        "reasoning_correct": (
            result.get(
                "reasoning_correct"
            )
        ),

        "judge_evidence": (
            result.get(
                "judge_evidence"
            )
        ),

        "parse_status": (
            result.get(
                "parse_status"
            )
        ),

        "raw_output": (
            result.get(
                "raw_output"
            )
        ),

        "finish_reason": (
            result.get(
                "finish_reason"
            )
        ),

        "error": (
            result.get(
                "error"
            )
        ),

        "time_seconds": (
            result.get(
                "time_seconds"
            )
        )
    }

    # ========================================================
    # 7. SAVE AFTER EVERY ROW
    # ========================================================

    new_row_df = pd.DataFrame(
        [result_row]
    )

    if results_df.empty:

        results_df = (
            new_row_df.copy()
        )

    else:

        results_df = pd.concat(
            [
                results_df,
                new_row_df
            ],
            ignore_index=True
        )

    # Ensure string index
    results_df["index"] = (
        results_df["index"]
        .astype(str)
    )

    # Remove duplicates
    results_df = (
        results_df
        .drop_duplicates(
            subset=["index"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    # ========================================================
    # IMMEDIATE CHECKPOINT
    # ========================================================

    results_df.to_csv(
        CHECKPOINT_PATH,
        index=False
    )

    print()
    print(
        "✅ ROW SAVED"
    )

    print(
        "Index:",
        index
    )

    print(
        "Answer:",
        result.get(
            "answer_correct"
        )
    )

    print(
        "Reasoning:",
        result.get(
            "reasoning_correct"
        )
    )

    print(
        "Parse:",
        result.get(
            "parse_status"
        )
    )

    print(
        "Checkpoint rows:",
        len(results_df)
    )

    # ========================================================
    # SMALL PAUSE
    # ========================================================

    time.sleep(1)

# ============================================================
# 8. FINAL CHECKPOINT SAVE
# ============================================================

results_df.to_csv(
    CHECKPOINT_PATH,
    index=False
)

print()
print("=" * 80)
print("CHECKPOINT STATUS")
print("=" * 80)

print(
    "Completed:",
    len(results_df),
    "/",
    EVAL_SIZE
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

# ============================================================
# 9. FINAL FILE ONLY WHEN ALL 300 ARE DONE
# ============================================================

if len(results_df) >= EVAL_SIZE:

    results_df.to_csv(
        FINAL_RESULTS_PATH,
        index=False
    )

    print()
    print("=" * 80)
    print("🎯 ALL 300 ROWS COMPLETE")
    print("=" * 80)

    print(
        "Final results:",
        FINAL_RESULTS_PATH
    )

else:

    print()
    print("=" * 80)
    print("EVALUATION PAUSED")
    print("=" * 80)

    print(
        "Completed:",
        len(results_df),
        "/",
        EVAL_SIZE
    )

    print(
        "Run this same cell again "
        "when the Groq quota resets."
    )

# ============================================================
# 10. CURRENT SUMMARY
# ============================================================

if not results_df.empty:

    print()
    print("=" * 80)
    print("CURRENT RESULT SUMMARY")
    print("=" * 80)

    print("\nParse status:")

    print(
        results_df[
            "parse_status"
        ].value_counts(
            dropna=False
        )
    )

    successful = results_df[
        results_df[
            "parse_status"
        ] == "OK"
    ].copy()

    if len(successful) > 0:

        valid_cases = successful[
            (
                successful[
                    "question_valid"
                ] == True
            )
            &
            (
                successful[
                    "reference_answer_correct"
                ] == True
            )
        ].copy()

        print(
            "\nSuccessfully judged:",
            len(successful)
        )

        print(
            "Valid evaluation cases:",
            len(valid_cases)
        )

        if len(valid_cases) > 0:

            answer_accuracy = (
                valid_cases[
                    "answer_correct"
                ].mean()
                * 100
            )

            reasoning_accuracy = (
                valid_cases[
                    "reasoning_correct"
                ].mean()
                * 100
            )

            print(
                f"\nCurrent answer accuracy: "
                f"{answer_accuracy:.2f}%"
            )

            print(
                f"Current reasoning accuracy: "
                f"{reasoning_accuracy:.2f}%"
            )

FIXED 300-ROW SAMPLE
Rows: 300
Unique indices: 300

EXISTING CHECKPOINT FOUND
Completed rows in checkpoint: 50

RESUME STATUS
Total sample: 300
Already completed: 50
Remaining: 250

Evaluating 51/300
Index: 29067
Attempt 1/1 failed:
Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01ks5qq6p0epv9qmpweb3en0h8` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197262, Requested 5435. Please try again in 19m25.104s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

🚨 GROQ RATE LIMIT DETECTED
Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01ks5qq6p0epv9qmpweb3en0h8` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197262, Requested 5435. Please try again in 19m25.104s. Need more tokens? Upgrade to Dev Tier today at https://console.gro

In [2]:
import os

print(
    os.path.exists(
        "/kaggle/working/"
        "llama_groq_random_300_checkpoint.csv"
    )
)

False
